# 1b. Генерация датасета из Magellan Data Repository

Этот пайплайн загружает реальные бенчмарк-датасеты для entity matching из
[Magellan Data Repository](https://sites.google.com/site/anhaidgroup/useful-stuff/the-magellan-data-repository)
и конвертирует их в формат, совместимый с Pipeline 01 (синтетические данные).

Процесс:
1. Загрузка конфига
2. Скачивание и парсинг датасетов (DeepMatcher формат)
3. Конвертация в TablePairData + SM данные (LLM описания + эмбеддинги)
4. Визуальный и статистический анализ

Выходные данные полностью совместимы с Pipeline 02/03/04.

## 1b.1 Загрузка конфигурации

In [ ]:
import json
import os
import sys
import logging

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from pathlib import Path

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger(__name__)

CONFIG_PATH = 'config.json'

with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
    full_config = json.load(f)

magellan_params = full_config['magellan']
print(f"Ollama host: {magellan_params['ollama_host']}")
print(f"LLM: {magellan_params['llm_model']}")
print(f"Embedding: {magellan_params['embedding_model']}")
print(f"Датасеты: {magellan_params['datasets']}")
print(f"Output: {magellan_params['output_dir']}/")

## 1b.2 Доступные датасеты Magellan

In [ ]:
from table_unifier.data_generation import MagellanConfig, MagellanDatasetLoader, MAGELLAN_DATASETS

print("Доступные датасеты Magellan Data Repository:")
print("=" * 60)
for name, info in MAGELLAN_DATASETS.items():
    marker = "✓" if name in magellan_params['datasets'] else " "
    print(f"  [{marker}] {name:25s} — {info['description']} (domain: {info['domain']})")
print(f"\nВыбрано: {len(magellan_params['datasets'])} датасетов")

## 1b.3 Загрузка и конвертация датасетов

In [ ]:
magellan_config = MagellanConfig(**magellan_params)

print(f"Датасеты: {magellan_config.datasets}")
print(f"Max пар на датасет: {magellan_config.max_pairs_per_dataset or 'все'}")
print(f"Output: {magellan_config.output_dir}/")

In [ ]:
loader = MagellanDatasetLoader(magellan_config)
loader.load_and_convert()

stats = loader.get_dataset_stats()
print(f"\nИтого пар: {stats['total_pairs']}")
for split, info in stats['splits'].items():
    if info['count'] > 0:
        print(f"  {split}: {info['count']} пар, "
              f"avg rows A={info['avg_rows_a']:.1f}, B={info['avg_rows_b']:.1f}, "
              f"avg dups={info['avg_duplicates']:.1f}")

## 1b.4 Статистический анализ SM датасета

In [ ]:
# Загрузка SM датасета
sm_dataset_path = os.path.join(magellan_config.output_dir, magellan_config.sm_dataset_file)
sm_metadata_path = os.path.join(magellan_config.output_dir, magellan_config.sm_metadata_file)

with open(sm_dataset_path, 'r', encoding='utf-8') as f:
    sm_data = json.load(f)

with open(sm_metadata_path, 'r', encoding='utf-8') as f:
    sm_meta = json.load(f)

print(f"SM записей: {len(sm_data)}")
print(f"Уникальных base_name: {sm_meta['stats']['unique_base_names']}")
print(f"Размерность эмбеддинга: {len(sm_data[0]['embedding'])}")

In [ ]:
# Распределение классов (base_name)
class_counts = Counter(entry['base_name'] for entry in sm_data)
class_df = pd.DataFrame(
    sorted(class_counts.items(), key=lambda x: x[1], reverse=True),
    columns=['base_name', 'count']
)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Барплот классов
axes[0].barh(class_df['base_name'], class_df['count'], color=sns.color_palette('viridis', len(class_df)))
axes[0].set_xlabel('Количество записей')
axes[0].set_title('Распределение классов (base_name) — Magellan')
axes[0].invert_yaxis()

# Boxplot количества по классам
axes[1].boxplot(class_df['count'], vert=True)
axes[1].set_title('Статистика по количеству записей в классе')
axes[1].set_ylabel('Количество записей')

plt.tight_layout()
plt.show()

print(f"\nСтатистика по классам:")
print(f"  min = {class_df['count'].min()}, max = {class_df['count'].max()}, "
      f"mean = {class_df['count'].mean():.1f}, median = {class_df['count'].median():.1f}")
print(f"  std = {class_df['count'].std():.1f}")

In [ ]:
# Распределение типов данных и источников
dtype_counts = Counter(entry['data_type'] for entry in sm_data)
source_counts = Counter(entry['source'] for entry in sm_data)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

labels, sizes = zip(*sorted(dtype_counts.items(), key=lambda x: x[1], reverse=True))
axes[0].pie(sizes, labels=labels, autopct='%1.1f%%', startangle=90)
axes[0].set_title('Распределение типов данных')

labels_s, sizes_s = zip(*sorted(source_counts.items(), key=lambda x: x[1], reverse=True))
axes[1].pie(sizes_s, labels=labels_s, autopct='%1.1f%%', startangle=90)
axes[1].set_title('Распределение по источнику (split)')

plt.tight_layout()
plt.show()

In [ ]:
# Анализ эмбеддингов: нормы и распределение
embeddings = np.array([entry['embedding'] for entry in sm_data])
norms = np.linalg.norm(embeddings, axis=1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(norms, bins=50, color='coral', edgecolor='white')
axes[0].set_xlabel('L2 норма эмбеддинга')
axes[0].set_title('Распределение норм эмбеддингов')
axes[0].axvline(np.mean(norms), color='red', linestyle='--', label=f'mean={np.mean(norms):.2f}')
axes[0].legend()

axes[1].bar(range(50), np.mean(embeddings[:, :50], axis=0), color='teal')
axes[1].set_xlabel('Компонента')
axes[1].set_title('Средние значения первых 50 компонент')

sample_flat = embeddings[:100].flatten()
axes[2].hist(sample_flat, bins=100, color='mediumpurple', edgecolor='white')
axes[2].set_xlabel('Значение компоненты')
axes[2].set_title('Распределение значений (100 эмбеддингов)')

plt.tight_layout()
plt.show()

print(f"Размерность: {embeddings.shape[1]}")
print(f"Нормы: min={norms.min():.4f}, max={norms.max():.4f}, mean={norms.mean():.4f}, std={norms.std():.4f}")

In [ ]:
# Матрица косинусных сходств средних эмбеддингов по классам
from sklearn.metrics.pairwise import cosine_similarity

class_embeddings = {}
for entry in sm_data:
    class_embeddings.setdefault(entry['base_name'], []).append(entry['embedding'])

class_names_sorted = sorted(class_embeddings.keys())
mean_embeddings = np.array([np.mean(class_embeddings[c], axis=0) for c in class_names_sorted])

sim_matrix = cosine_similarity(mean_embeddings)

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(sim_matrix, cmap='RdYlBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(class_names_sorted)))
ax.set_yticks(range(len(class_names_sorted)))
ax.set_xticklabels(class_names_sorted, rotation=45, ha='right')
ax.set_yticklabels(class_names_sorted)
ax.set_title('Косинусное сходство средних эмбеддингов по классам (Magellan, до обучения SM)')
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

## 1b.5 Статистический анализ ER датасета

In [ ]:
# Анализ ER пар
er_raw_dir = os.path.join(magellan_config.output_dir, magellan_config.er_raw_dir)
er_stats = []

for split in ['train', 'val', 'test']:
    split_dir = os.path.join(er_raw_dir, split)
    if not os.path.exists(split_dir):
        continue

    meta_files = sorted(Path(split_dir).glob('*_meta.json'))
    for meta_file in meta_files:
        with open(meta_file, 'r', encoding='utf-8') as f:
            meta = json.load(f)
        meta['split'] = split
        er_stats.append(meta)

er_df = pd.DataFrame(er_stats)
print(f"Всего ER пар: {len(er_df)}")
print(f"\nПо сплитам:")
print(er_df.groupby('split')[['num_rows_a', 'num_rows_b', 'num_duplicates']].agg(['mean', 'min', 'max']).round(1))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Распределение количества строк в таблицах
all_rows = list(er_df['num_rows_a']) + list(er_df['num_rows_b'])
axes[0].hist(all_rows, bins=30, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_xlabel('Количество строк')
axes[0].set_ylabel('Частота')
axes[0].set_title('Распределение размеров таблиц (ER, Magellan)')
axes[0].axvline(np.mean(all_rows), color='red', linestyle='--', label=f'mean={np.mean(all_rows):.1f}')
axes[0].legend()

# Распределение количества дубликатов
axes[1].hist(er_df['num_duplicates'], bins=range(0, er_df['num_duplicates'].max() + 2),
             color='coral', edgecolor='white', alpha=0.8)
axes[1].set_xlabel('Количество дубликатов')
axes[1].set_ylabel('Частота')
axes[1].set_title('Распределение количества дубликатов в паре')
axes[1].axvline(er_df['num_duplicates'].mean(), color='red', linestyle='--',
                label=f'mean={er_df["num_duplicates"].mean():.1f}')
axes[1].legend()

# Доля дубликатов
dup_ratio = er_df['num_duplicates'] / ((er_df['num_rows_a'] + er_df['num_rows_b']) / 2)
axes[2].hist(dup_ratio, bins=30, color='mediumpurple', edgecolor='white', alpha=0.8)
axes[2].set_xlabel('Доля дубликатов')
axes[2].set_ylabel('Частота')
axes[2].set_title('Доля дубликатов от среднего размера таблицы')

plt.tight_layout()
plt.show()

In [ ]:
# Пример пары таблиц
example_split = 'train'
example_dir = os.path.join(er_raw_dir, example_split)
example_a = pd.read_csv(os.path.join(example_dir, 'pair_0000_table_a.csv'))
example_b = pd.read_csv(os.path.join(example_dir, 'pair_0000_table_b.csv'))

with open(os.path.join(example_dir, 'pair_0000_meta.json'), 'r', encoding='utf-8') as f:
    example_meta = json.load(f)

print(f"Таблица A: {example_a.shape}, столбцы: {list(example_a.columns)}")
print(f"Таблица B: {example_b.shape}, столбцы: {list(example_b.columns)}")
print(f"Дубликаты: {example_meta['num_duplicates']} пар")
print(f"Источник: {example_meta.get('source', 'N/A')}")

print("\n--- Таблица A (первые 5 строк) ---")
display(example_a.head())

print("\n--- Таблица B (первые 5 строк) ---")
display(example_b.head())

## 1b.6 Визуализация эмбеддингов (t-SNE до обучения SM)

In [ ]:
from sklearn.manifold import TSNE

# t-SNE на сырых эмбеддингах (до обучения ProjectionHead)
embeddings = np.array([entry['embedding'] for entry in sm_data])
labels = [entry['base_name'] for entry in sm_data]

tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(sm_data) - 1), max_iter=1000)
tsne_result = tsne.fit_transform(embeddings)

unique_labels = sorted(set(labels))
cmap = plt.cm.get_cmap('tab20', len(unique_labels))

fig, ax = plt.subplots(figsize=(14, 10))
for i, label in enumerate(unique_labels):
    mask = np.array(labels) == label
    ax.scatter(tsne_result[mask, 0], tsne_result[mask, 1],
               c=[cmap(i)], label=label, alpha=0.6, s=20)

ax.set_title('t-SNE: сырые эмбеддинги столбцов — Magellan (до обучения SM)', fontsize=14)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()

## 1b.7 Итоговая сводка

In [ ]:
print("=" * 60)
print("ИТОГОВАЯ СВОДКА ДАТАСЕТА (Magellan Data Repository)")
print("=" * 60)
print(f"\nИсточник: Magellan Data Repository")
print(f"Датасеты: {magellan_config.datasets}")
print(f"\nSchema Matching:")
print(f"  Записей: {len(sm_data)}")
print(f"  Классов: {sm_meta['stats']['unique_base_names']}")
print(f"  Размерность эмбеддинга: {embeddings.shape[1]}")
print(f"  Файл: {sm_dataset_path}")
print(f"\nEntity Resolution:")
print(f"  Пар таблиц: {len(er_df)}")
for split in ['train', 'val', 'test']:
    split_df = er_df[er_df['split'] == split]
    if len(split_df) > 0:
        print(f"    {split}: {len(split_df)} пар, "
              f"avg rows={split_df['num_rows_a'].mean():.1f}/{split_df['num_rows_b'].mean():.1f}, "
              f"avg dups={split_df['num_duplicates'].mean():.1f}")
print(f"  Директория: {er_raw_dir}")
print(f"\nСледующий шаг: Pipeline 2 — Обучение Schema Matching")
print(f"  (укажите output_dir='{magellan_config.output_dir}' для использования Magellan данных)")